# Testing with Pretrained Models


## Frameworks

In [ ]:
import os
import json
import base64
import io
import math
import random
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.utils import save_image, make_grid
from PIL import Image as PILImage
import matplotlib.pyplot as plt
import cv2

import sys
sys.path.append(str(Path.cwd().parent))

from utils import DeepFakeDataset, set_global_seed, dcganFormat  # dataset usado nas experiências anteriores

c:\Users\hasht\anaconda3\envs\adversarial\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config & Setup

In [ ]:
# =========================================================
# Config — g6 face-crop + regularização leve do D
# =========================================================
RUN_NAME = "g9_pretrained_gans"

REAL_DIR = "../../deepfake_data/wiki"
BASE_OUTPUT_DIR = "outputs"
OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, RUN_NAME)
SAMPLES_DIR = os.path.join(OUTPUT_DIR, "samples")
CHECKPOINTS_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")
METRICS_DIR = os.path.join(OUTPUT_DIR, "metrics")


IMAGE_SIZE = 64
BATCH_SIZE = 64
LATENT_DIM = 100
NGF = 64
NDF = 64
NUM_CHANNELS = 3

NUM_EPOCHS = 400
LR_G = 2e-4
LR_D = 1e-4
BETA1 = 0.5
SEED = 42
NUM_WORKERS = 4

# g6: one-sided label smoothing apenas para imagens reais.
# O alvo real do discriminador é amostrado em [0.85, 1.0].
REAL_LABEL_LOW = 0.85
REAL_LABEL_HIGH = 1.0
FAKE_LABEL = 0.0
GEN_TARGET_LABEL = 1.0

# g6: instance noise. Ajuda a impedir que D se torne perfeito demasiado cedo.
INSTANCE_NOISE_STD_INIT = 0.05
INSTANCE_NOISE_DECAY_EPOCHS = 80

# g6: dropout leve no discriminador.
DROPOUT_P = 0.20

# Face crop
USE_FACE_CROP = True
FACE_CROP_MARGIN = 0.45
FACE_DETECT_SCALE_FACTOR = 1.1
FACE_DETECT_MIN_NEIGHBORS = 5
MIN_FACE_SIZE = 20

START_FOLD = 0
END_FOLD = 5
INTERVAL = True

# Logging / avaliação
SAMPLE_EVERY = 25
CHECKPOINT_EVERY = 10
FID_EVERY = 50
FID_NUM_IMAGES = 300      # usar 300 se for demasiado lento; 1000+ é melhor para avaliação final
COMPUTE_KID = True
SAVE_JSONL_EVERY = None

USE_EMA = True
EMA_DECAY = 0.999

ALWAYS_SAVE_EPOCHS = {1, NUM_EPOCHS}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"

for folder in [OUTPUT_DIR, SAMPLES_DIR, CHECKPOINTS_DIR, PLOTS_DIR, METRICS_DIR]:
    os.makedirs(folder, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

print("Run:", RUN_NAME)
print("Device:", DEVICE)
print("Output dir:", OUTPUT_DIR)
print("Folds:", START_FOLD, "->", END_FOLD)

Run: g7_pretrained_models
Device: cuda
Output dir: outputs\g7_pretrained_models
Folds: 0 -> 5


## Utils

In [ ]:

def denorm_to_01(x):
    return ((x + 1) / 2).clamp(0, 1)


def should_run(epoch, every):
    if every is None:
        return False
    return epoch in ALWAYS_SAVE_EPOCHS or epoch % every == 0


def save_generated_samples_png(epoch, generator, noise, out_dir=SAMPLES_DIR, prefix="fixed_noise", nrow=8):
    was_training = generator.training
    generator.eval()
    with torch.no_grad():
        fake_images = denorm_to_01(generator(noise).detach().cpu())
        grid = make_grid(fake_images, nrow=nrow, padding=2, normalize=False)
        path = os.path.join(out_dir, f"{prefix}_epoch_{epoch:03d}.png")
        save_image(grid, path)
    if was_training:
        generator.train()
    return path


def save_samples_jsonl(epoch, generator, noise, path, nrow=8):
    was_training = generator.training
    generator.eval()
    with torch.no_grad():
        fake = denorm_to_01(generator(noise).detach().cpu())
        grid = make_grid(fake, nrow=nrow, padding=2, normalize=False)
        arr = (grid.permute(1, 2, 0).numpy() * 255).clip(0, 255).astype("uint8")
        buf = io.BytesIO()
        PILImage.fromarray(arr).save(buf, format="PNG", optimize=True)
        b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    if was_training:
        generator.train()

    with open(path, "a") as f:
        f.write(json.dumps({"epoch": epoch, "format": "png", "grid_b64": b64}) + "\n")

## Reading Data

In [ ]:
dataset = DeepFakeDataset(
    img_dir=REAL_DIR,
    label=1,
    transform=dcganFormat(IMAGE_SIZE),,
    range_folds=[START_FOLD, END_FOLD],
    interval=INTERVAL,
    image_only=True,
)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True,
)

print(f"Total de imagens reais: {len(dataset)}")
try:
    print("Folds usados:", dataset.fold_names[:5], "...", dataset.fold_names[-5:])
except Exception:
    print("Dataset carregado.")

Building ../../deepfake_data/wiki dataset with 5 folds: 100%|██████████| 5/5 [00:00<00:00, 832.47it/s]

Total de imagens reais: 1467
Folds usados: ['00', '01', '02', '03', '04'] ... ['00', '01', '02', '03', '04']


## Pretrained Models

In [ ]:
# ─── CELL 1: Pretrained GAN setup ────────────────────────────────────────────
import torch.nn as nn
import torch.nn.functional as F

class PGANWrapper(nn.Module):
    def __init__(self, pgan_model, target_size=64):
        super().__init__()
        self._pgan = pgan_model
        self.target_size = target_size

    def forward(self, z):
        imgs = self._pgan.test(z)
        if imgs.shape[-1] != self.target_size:
            imgs = F.interpolate(imgs, size=(self.target_size, self.target_size),
                                 mode="bilinear", align_corners=False)
        return imgs

print("Loading PGAN from torch.hub …")
_pgan_model = torch.hub.load(
    "facebookresearch/pytorch_GAN_zoo:hub",
    "PGAN", model_name="celeba_cropped",
    pretrained=True, useGPU=(DEVICE.type == "cuda"),
)

netG = PGANWrapper(_pgan_model, target_size=IMAGE_SIZE)
netG.eval()

fixed_noise, _ = _pgan_model.buildNoiseData(BATCH_SIZE)
fixed_noise = fixed_noise.to(DEVICE)
print(f"Ready. fixed_noise: {tuple(fixed_noise.shape)}")


# ─── CELL 2: Calibration (unchanged) ─────────────────────────────────────────
from tqdm.auto import tqdm
import pandas as pd
import math
import io, base64, json
from PIL import Image as PILImage

TARGET_MB = 90
TARGET_BYTES = TARGET_MB * 1024 * 1024

def _sample_line_bytes(generator, noise, nrow=8):
    generator.eval()
    with torch.no_grad():
        fake = generator(noise).detach().cpu()
        fake = (fake + 1) / 2
        grid = make_grid(fake, nrow=nrow, padding=2, normalize=False)
        arr = (grid.permute(1, 2, 0).numpy() * 255).clip(0, 255).astype("uint8")
        buf = io.BytesIO()
        PILImage.fromarray(arr).save(buf, format="PNG", optimize=True)
        b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    generator.train()
    line = json.dumps({"epoch": 0, "format": "png", "grid_b64": b64}) + "\n"
    return len(line.encode("utf-8"))

frame_bytes = _sample_line_bytes(netG, fixed_noise)
total_bytes = frame_bytes * NUM_EPOCHS

EPOCHS_PER_SPLIT = max(1, TARGET_BYTES // frame_bytes)
NUM_SPLITS = math.ceil(NUM_EPOCHS / EPOCHS_PER_SPLIT)
last_epochs = NUM_EPOCHS - (NUM_SPLITS - 1) * EPOCHS_PER_SPLIT

full_mb = (EPOCHS_PER_SPLIT * frame_bytes) / (1024 * 1024)
last_mb = (last_epochs * frame_bytes) / (1024 * 1024)

print("Sample-file calibration")
print(f"  frame size:       ~{frame_bytes/1024:>7.1f} KB")
print(f"  projected total:  ~{total_bytes/1024/1024:>7.1f} MB  ({NUM_EPOCHS} epochs)")
print(f"  → {NUM_SPLITS} files: {NUM_SPLITS-1} × ~{full_mb:.1f} MB + 1 × ~{last_mb:.1f} MB")

JSON_OUT_DIR = os.path.join(OUTPUT_DIR, "json_samples")
os.makedirs(JSON_OUT_DIR, exist_ok=True)
SAMPLE_FILES = [
    os.path.join(JSON_OUT_DIR, f"samples_{k+1:02d}.jsonl")
    for k in range(NUM_SPLITS)
]
for p in SAMPLE_FILES: open(p, "w").close()

def sample_file_for(epoch):
    return SAMPLE_FILES[min((epoch - 1) // EPOCHS_PER_SPLIT, NUM_SPLITS - 1)]


# ─── CELL 3: Generation loop ──────────────────────────────────────────────────
history = []
history_path = os.path.join(METRICS_DIR, "generation_history.csv")

save_generated_samples_png(0, netG, fixed_noise, prefix="fixed_noise_initial")

epoch_bar = tqdm(range(1, NUM_EPOCHS + 1), desc="Sampling", unit="epoch")
for epoch in epoch_bar:
    frame_noise, _ = _pgan_model.buildNoiseData(BATCH_SIZE)
    frame_noise = frame_noise.to(DEVICE)

    save_samples_jsonl(epoch, netG, frame_noise, sample_file_for(epoch))

    if should_run(epoch, SAMPLE_EVERY):
        path = save_generated_samples_png(epoch, netG, fixed_noise, prefix="fixed_noise")
        print(f"[epoch {epoch}] PNG: {path}")

    history.append({"run_name": RUN_NAME, "epoch": epoch})

    if epoch % 50 == 0:
        pd.DataFrame(history).to_csv(history_path, index=False)

    epoch_bar.set_postfix(file=os.path.basename(sample_file_for(epoch)))

pd.DataFrame(history).to_csv(history_path, index=False)
print(f"Done. {NUM_EPOCHS} frames → {JSON_OUT_DIR}")

G: 4,103,075  |  D: 2,765,633
  frame ~2KB | saving every 1 ep | 1 file(s)


FastGAN:   8%|▊         | 16/200 [06:14<1:11:49, 23.42s/it]


KeyboardInterrupt: 

### Style2Gan

In [ ]:
```python
# ─── CELL 1: Pretrained GAN setup ────────────────────────────────────────────
import torch.nn as nn
import torch.nn.functional as F

class PGANWrapper(nn.Module):
    def __init__(self, pgan_model, target_size=64):
        super().__init__()
        self._pgan = pgan_model
        self.target_size = target_size

    def forward(self, z):
        imgs = self._pgan.test(z)
        if imgs.shape[-1] != self.target_size:
            imgs = F.interpolate(imgs, size=(self.target_size, self.target_size),
                                 mode="bilinear", align_corners=False)
        return imgs

print("Loading PGAN from torch.hub …")
_pgan_model = torch.hub.load(
    "facebookresearch/pytorch_GAN_zoo:hub",
    "PGAN", model_name="celeba_cropped",
    pretrained=True, useGPU=(DEVICE.type == "cuda"),
)

netG = PGANWrapper(_pgan_model, target_size=IMAGE_SIZE)
netG.eval()

fixed_noise, _ = _pgan_model.buildNoiseData(BATCH_SIZE)
fixed_noise = fixed_noise.to(DEVICE)
print(f"Ready. fixed_noise: {tuple(fixed_noise.shape)}")


# ─── CELL 2: Calibration (unchanged) ─────────────────────────────────────────
from tqdm.auto import tqdm
import pandas as pd
import math
import io, base64, json
from PIL import Image as PILImage

TARGET_MB = 90
TARGET_BYTES = TARGET_MB * 1024 * 1024

def _sample_line_bytes(generator, noise, nrow=8):
    generator.eval()
    with torch.no_grad():
        fake = generator(noise).detach().cpu()
        fake = (fake + 1) / 2
        grid = make_grid(fake, nrow=nrow, padding=2, normalize=False)
        arr = (grid.permute(1, 2, 0).numpy() * 255).clip(0, 255).astype("uint8")
        buf = io.BytesIO()
        PILImage.fromarray(arr).save(buf, format="PNG", optimize=True)
        b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    generator.train()
    line = json.dumps({"epoch": 0, "format": "png", "grid_b64": b64}) + "\n"
    return len(line.encode("utf-8"))

frame_bytes = _sample_line_bytes(netG, fixed_noise)
total_bytes = frame_bytes * NUM_EPOCHS

EPOCHS_PER_SPLIT = max(1, TARGET_BYTES // frame_bytes)
NUM_SPLITS = math.ceil(NUM_EPOCHS / EPOCHS_PER_SPLIT)
last_epochs = NUM_EPOCHS - (NUM_SPLITS - 1) * EPOCHS_PER_SPLIT

full_mb = (EPOCHS_PER_SPLIT * frame_bytes) / (1024 * 1024)
last_mb = (last_epochs * frame_bytes) / (1024 * 1024)

print("Sample-file calibration")
print(f"  frame size:       ~{frame_bytes/1024:>7.1f} KB")
print(f"  projected total:  ~{total_bytes/1024/1024:>7.1f} MB  ({NUM_EPOCHS} epochs)")
print(f"  → {NUM_SPLITS} files: {NUM_SPLITS-1} × ~{full_mb:.1f} MB + 1 × ~{last_mb:.1f} MB")

JSON_OUT_DIR = os.path.join(OUTPUT_DIR, "json_samples")
os.makedirs(JSON_OUT_DIR, exist_ok=True)
SAMPLE_FILES = [
    os.path.join(JSON_OUT_DIR, f"samples_{k+1:02d}.jsonl")
    for k in range(NUM_SPLITS)
]
for p in SAMPLE_FILES: open(p, "w").close()

def sample_file_for(epoch):
    return SAMPLE_FILES[min((epoch - 1) // EPOCHS_PER_SPLIT, NUM_SPLITS - 1)]


# ─── CELL 3: Generation loop ──────────────────────────────────────────────────
history = []
history_path = os.path.join(METRICS_DIR, "generation_history.csv")

save_generated_samples_png(0, netG, fixed_noise, prefix="fixed_noise_initial")

epoch_bar = tqdm(range(1, NUM_EPOCHS + 1), desc="Sampling", unit="epoch")
for epoch in epoch_bar:
    frame_noise, _ = _pgan_model.buildNoiseData(BATCH_SIZE)
    frame_noise = frame_noise.to(DEVICE)

    save_samples_jsonl(epoch, netG, frame_noise, sample_file_for(epoch))

    if should_run(epoch, SAMPLE_EVERY):
        path = save_generated_samples_png(epoch, netG, fixed_noise, prefix="fixed_noise")
        print(f"[epoch {epoch}] PNG: {path}")

    history.append({"run_name": RUN_NAME, "epoch": epoch})

    if epoch % 50 == 0:
        pd.DataFrame(history).to_csv(history_path, index=False)

    epoch_bar.set_postfix(file=os.path.basename(sample_file_for(epoch)))

pd.DataFrame(history).to_csv(history_path, index=False)
print(f"Done. {NUM_EPOCHS} frames → {JSON_OUT_DIR}")
```

ValueError: betas must be either both floats or both Tensors